## 1. Memasang dan memeriksa Git

Git dapat diunduh dari situs resmi git-scm.com/downloads. Di Google Colab, Git sudah terpasang sehingga cukup memeriksa pemasangannya dengan perintah `git --version`.

Sel pertama membuat fungsi bantu `sh()` yang menjalankan perintah dan menampilkannya seperti di terminal (baris berawalan `$` adalah perintah yang dijalankan).

In [1]:
import os, shlex, shutil, subprocess, stat, tempfile
from pathlib import Path

# Di Colab folder kerja adalah /content.
BASE = Path("/content") if Path("/content").is_dir() else Path.cwd()
REPO = BASE / "catatan-git"


def sh(perintah, cwd=None, env=None):
    """Jalankan perintah di terminal dan tampilkan seperti Git Bash."""
    for baris in perintah.strip().splitlines():
        print(f"$ {baris}")
    lingkungan = os.environ.copy()
    lingkungan.update({"GIT_PAGER": "cat", "GIT_EDITOR": "true", "GIT_TERMINAL_PROMPT": "0"})
    if env:
        lingkungan.update(env)
    folder = cwd if cwd else (REPO if REPO.exists() else BASE)
    hasil = subprocess.run(
        perintah, shell=True, executable="/bin/bash", cwd=str(folder),
        capture_output=True, text=True, env=lingkungan,
    )
    if hasil.stdout.strip():
        print(hasil.stdout.rstrip())
    if hasil.stderr.strip():
        print(hasil.stderr.rstrip())
    if hasil.returncode != 0:
        print(f"[perintah berhenti dengan kode {hasil.returncode}]")
    print()
    return hasil


def tulis(nama, isi, mode="w"):
    """Tulis isi ke berkas di dalam repositori."""
    with open(REPO / nama, mode, encoding="utf-8") as f:
        f.write(isi)


sh("git --version", cwd=BASE)

$ git --version
git version 2.43.0



CompletedProcess(args='git --version', returncode=0, stdout='git version 2.43.0\n', stderr='')

## 2. Konfigurasi awal

Git perlu mengetahui identitas pengguna karena setiap commit mencatat nama dan surel penulisnya. Konfigurasi ini cukup dilakukan sekali.

- Perintah pertama dan kedua mengatur nama dan surel.
- Perintah ketiga mengatur nama cabang awal menjadi `main`.
- Perintah terakhir menampilkan seluruh konfigurasi yang aktif.

In [3]:
sh('git config --global user.name "M. Awaludin Ramadan"', cwd=BASE)
sh('git config --global user.email "awalramadhan995@gmail.com"', cwd=BASE)
sh("git config --global init.defaultBranch main", cwd=BASE)
sh("git config --list", cwd=BASE)

$ git config --global user.name "M. Awaludin Ramadan"

$ git config --global user.email "awalramadhan995@gmail.com"

$ git config --global init.defaultBranch main

$ git config --list
user.name=M. Awaludin Ramadan
user.email=awalramadhan995@gmail.com
init.defaultbranch=main



CompletedProcess(args='git config --list', returncode=0, stdout='user.name=M. Awaludin Ramadan\nuser.email=awalramadhan995@gmail.com\ninit.defaultbranch=main\n', stderr='')

## 3. Membuat repositori lokal

Dibuat folder baru bernama `catatan-git`, kemudian diubah menjadi repositori Git.

- `mkdir` membuat folder.
- `cd` berpindah ke dalam folder (pada notebook ini semua perintah berikutnya otomatis dijalankan di dalam folder `catatan-git`).
- `git init` membuat repositori, ditandai munculnya folder tersembunyi `.git`.
- `git status` menampilkan keadaan repositori. Pada kondisi awal, `git status` menyatakan belum ada commit dan tidak ada berkas yang dilacak.

In [ ]:
if REPO.exists():
    shutil.rmtree(REPO)

sh("mkdir catatan-git", cwd=BASE)
sh("cd catatan-git", cwd=BASE)
sh("git init")
sh("git status")

## 4. Membuat README.md

`README.md` adalah berkas penjelasan proyek yang otomatis tampil di halaman utama repositori. Ekstensi `.md` menunjukkan isinya ditulis dengan format Markdown. Berikut berkas README.md sederhana untuk repositori ini.

In [ ]:
tulis("README.md", """# Catatan Belajar Git dan GitHub

Repositori ini berisi catatan belajar mata kuliah
Pengolahan Citra Digital.

## Isi Repositori
- `catatan.txt` : catatan pertemuan pertama
- `perintah-git.md` : rangkuman perintah dasar Git

## Penulis
M. Awaludin Ramadan - 24260027
""")

sh("cat README.md")

## 5. `git add`, `git commit`, dan `git log`

Alur yang paling sering dipakai: memilih perubahan, menyimpannya sebagai titik simpan, lalu (pada bagian 9) mengirimkannya ke GitHub. Dibuat berkas `catatan.txt` di dalam folder repositori.

1. `echo ... > catatan.txt` membuat berkas teks berisi satu kalimat.
2. `git status` yang pertama menampilkan `catatan.txt` sebagai berkas baru yang belum dilacak (*untracked*).
3. `git add catatan.txt` memasukkan berkas ke *staging area*.
4. `git status` yang kedua menunjukkan berkas sudah siap di-commit (*changes to be committed*).
5. `git commit -m "..."` menyimpan isi staging area sebagai commit baru dengan pesan yang ditulis di antara tanda petik.

In [4]:
sh('echo contoh"Catatan hari pertama belajar Git" > catatan.txt')
sh("git status")
sh("git add catatan.txt")
sh("git status")
sh('git commit -m "Menambahkan catatan.txt"')

$ echo contoh"Catatan hari pertama belajar Git" > catatan.txt

$ git status
fatal: not a git repository (or any of the parent directories): .git
[perintah berhenti dengan kode 128]

$ git add catatan.txt
fatal: not a git repository (or any of the parent directories): .git
[perintah berhenti dengan kode 128]

$ git status
fatal: not a git repository (or any of the parent directories): .git
[perintah berhenti dengan kode 128]

$ git commit -m "Menambahkan catatan.txt"
fatal: not a git repository (or any of the parent directories): .git
[perintah berhenti dengan kode 128]



CompletedProcess(args='git commit -m "Menambahkan catatan.txt"', returncode=128, stdout='', stderr='fatal: not a git repository (or any of the parent directories): .git\n')

`README.md` yang dibuat pada bagian 4 juga dimasukkan dengan alur yang sama. Setelah itu riwayat dilihat dengan `git log --oneline` (satu baris per commit) dan `git diff` (perubahan yang belum di-stage; kosong karena semua perubahan sudah di-commit).

**Tips pesan commit:** tulis singkat dan jelas, hindari pesan seperti `update` atau `coba`, dan satu commit sebaiknya berisi satu kelompok perubahan yang saling berkaitan.

In [5]:
sh("git add README.md")
sh('git commit -m "Menambahkan README.md"')
sh("git log --oneline")
sh("git diff")

$ git add README.md
fatal: not a git repository (or any of the parent directories): .git
[perintah berhenti dengan kode 128]

$ git commit -m "Menambahkan README.md"
fatal: not a git repository (or any of the parent directories): .git
[perintah berhenti dengan kode 128]

$ git log --oneline
fatal: not a git repository (or any of the parent directories): .git
[perintah berhenti dengan kode 128]

$ git diff
usage: git diff --no-index [<options>] <path> <path>

Diff output format options
    -p, --patch           generate patch
    -s, --no-patch        suppress diff output
    -u                    generate patch
    -U, --unified[=<n>]   generate diffs with <n> lines context
    -W, --[no-]function-context
                          generate diffs with <n> lines context
    --raw                 generate the diff in raw format
    --patch-with-raw      synonym for '-p --raw'
    --patch-with-stat     synonym for '-p --stat'
    --numstat             machine friendly --stat
    --shortsta

CompletedProcess(args='git diff', returncode=129, stdout='', stderr='warning: Not a git repository. Use --no-index to compare two paths outside a working tree\nusage: git diff --no-index [<options>] <path> <path>\n\nDiff output format options\n    -p, --patch           generate patch\n    -s, --no-patch        suppress diff output\n    -u                    generate patch\n    -U, --unified[=<n>]   generate diffs with <n> lines context\n    -W, --[no-]function-context\n                          generate diffs with <n> lines context\n    --raw                 generate the diff in raw format\n    --patch-with-raw      synonym for \'-p --raw\'\n    --patch-with-stat     synonym for \'-p --stat\'\n    --numstat             machine friendly --stat\n    --shortstat           output only the last line of --stat\n    -X, --dirstat[=<param1,param2>...]\n                          output the distribution of relative amount of changes for each sub-directory\n    --cumulative          synonym for -

## 6. Branch dan merge

*Branch* adalah jalur pengembangan yang terpisah, memungkinkan pengguna bekerja pada fitur atau perbaikan tanpa memengaruhi versi utama. Analoginya seperti membuat salinan naskah untuk dicoret-coret: jika hasilnya bagus, perubahan dimasukkan ke naskah utama.

Sebagai persiapan, dibuat berkas `judul.txt` pada cabang `main`.

In [ ]:
sh('echo "Judul: Catatan" > judul.txt')
sh("git add judul.txt")
sh('git commit -m "Menambahkan judul.txt"')

### a. Perintah dasar branch

- `git branch` menampilkan daftar cabang dan menandai cabang aktif.
- `git branch fitur-judul` membuat cabang `fitur-judul`.
- `git switch fitur-judul` berpindah ke cabang tersebut.
- `git switch -c fitur-lain` membuat sekaligus berpindah ke cabang `fitur-lain`.

In [6]:
sh("git branch")
sh("git branch fitur-judul")
sh("git switch fitur-judul")
sh("git switch -c fitur-lain")
sh("git branch")

$ git branch
fatal: not a git repository (or any of the parent directories): .git
[perintah berhenti dengan kode 128]

$ git branch fitur-judul
fatal: not a git repository (or any of the parent directories): .git
[perintah berhenti dengan kode 128]

$ git switch fitur-judul
fatal: not a git repository (or any of the parent directories): .git
[perintah berhenti dengan kode 128]

$ git switch -c fitur-lain
fatal: not a git repository (or any of the parent directories): .git
[perintah berhenti dengan kode 128]

$ git branch
fatal: not a git repository (or any of the parent directories): .git
[perintah berhenti dengan kode 128]



CompletedProcess(args='git branch', returncode=128, stdout='', stderr='fatal: not a git repository (or any of the parent directories): .git\n')

### b. Merge

Setelah pekerjaan di cabang selesai dan sudah di-commit, perubahan digabungkan ke cabang utama: berpindah ke cabang tujuan (`main`), menggabungkan cabang dengan `git merge`, lalu mengirim hasilnya ke GitHub dengan `git push origin main` (dilakukan pada bagian 9).

Pada cabang `fitur-lain` dibuat berkas `perintah-git.md` (rangkuman perintah dasar Git pada Tabel 2.4). Karena cabang `main` tidak berubah sejak cabang dibuat, Git cukup menggeser penunjuk ke depan, yang disebut ***fast-forward***.

In [11]:
perintah = [
    ("git --version", "Menampilkan versi Git yang terpasang.", "git --version"),
    ("git config", "Mengatur konfigurasi, misalnya nama dan email pengguna.", 'git config --global user.name "Nama"'),
    ("git init", "Membuat repositori Git baru di folder saat ini.", "git init"),
    ("git clone", "Menyalin repositori dari remote ke komputer.", "git clone https://github.com/user/repo.git"),
    ("git status", "Menampilkan status berkas (baru, berubah, atau sudah di-stage).", "git status"),
    ("git add", "Memasukkan berkas ke staging area.", "git add catatan.txt"),
    ("git commit", "Menyimpan isi staging area sebagai titik simpan baru.", 'git commit -m "Tambah catatan"'),
    ("git log", "Menampilkan riwayat commit.", "git log --oneline"),
    ("git diff", "Menampilkan perbedaan isi berkas.", "git diff"),
    ("git branch", "Melihat atau membuat cabang.", "git branch fitur-baru"),
    ("git switch", "Berpindah ke cabang lain.", "git switch fitur-baru"),
    ("git merge", "Menggabungkan cabang ke cabang aktif.", "git merge fitur-baru"),
    ("git remote", "Mengelola alamat remote repositori.", "git remote add origin URL"),
    ("git push", "Mengirim commit lokal ke remote.", "git push origin main"),
    ("git pull", "Mengambil dan menggabungkan perubahan dari remote.", "git pull origin main"),
    ("git fetch", "Mengambil data terbaru dari remote tanpa menggabungkan.", "git fetch origin"),
    ("git restore", "Mengembalikan berkas ke keadaan sebelumnya atau mengeluarkannya dari staging.", "git restore catatan.txt"),
    ("git revert", "Membuat commit baru yang membatalkan commit tertentu.", "git revert HEAD"),
]

isi = "# Perintah Dasar Git\n\n| Perintah | Fungsi | Contoh |\n|---|---|---|\n"
isi += "".join(f"| `{p}` | {f} | `{c}` |\n" for p, f, c in perintah)

# Memastikan folder REPO dibuat terlebih dahulu agar tidak terjadi FileNotFoundError
REPO.mkdir(parents=True, exist_ok=True)
tulis("perintah-git.md", isi)

sh("git add perintah-git.md")
sh('git commit -m "Menambahkan perintah-git.md"')
sh("git switch main")
sh("git merge fitur-lain")
sh("git log --oneline --graph --all")

$ git add perintah-git.md
fatal: not a git repository (or any of the parent directories): .git
[perintah berhenti dengan kode 128]

$ git commit -m "Menambahkan perintah-git.md"
fatal: not a git repository (or any of the parent directories): .git
[perintah berhenti dengan kode 128]

$ git switch main
fatal: not a git repository (or any of the parent directories): .git
[perintah berhenti dengan kode 128]

$ git merge fitur-lain
fatal: not a git repository (or any of the parent directories): .git
[perintah berhenti dengan kode 128]

$ git log --oneline --graph --all
fatal: not a git repository (or any of the parent directories): .git
[perintah berhenti dengan kode 128]



CompletedProcess(args='git log --oneline --graph --all', returncode=128, stdout='', stderr='fatal: not a git repository (or any of the parent directories): .git\n')

### c. Konflik penggabungan

Konflik terjadi ketika dua cabang mengubah baris yang sama pada berkas yang sama, sehingga Git tidak tahu versi mana yang harus dipakai. Git menghentikan penggabungan dan menandai berkas yang bermasalah dengan penanda `<<<<<<<`, `=======`, dan `>>>>>>>`.

Di sini baris judul pada `judul.txt` diubah di cabang `fitur-judul` dan juga di cabang `main`, lalu `fitur-judul` digabungkan ke `main`.

In [ ]:
# Perubahan di cabang fitur-judul
sh("git switch fitur-judul")
sh('echo "Judul: Catatan Belajar Git dan GitHub" > judul.txt')
sh('git commit -am "Mengubah judul di fitur-judul"')

# Perubahan pada baris yang sama di cabang main
sh("git switch main")
sh('echo "Judul: Catatan Git" > judul.txt')
sh('git commit -am "Mengubah judul di main"')

# Penggabungan -> konflik
sh("git merge fitur-judul")
sh("git status")
sh("cat judul.txt")

**Cara menyelesaikan konflik:**
1. Buka berkas yang berkonflik dan tentukan isi akhir yang diinginkan, apakah memakai salah satu versi atau menggabungkan keduanya.
2. Hapus seluruh penanda `<<<<<<<`, `=======`, dan `>>>>>>>`.
3. Jalankan `git add` pada berkas tersebut.
4. Jalankan `git commit` untuk menyelesaikan penggabungan.

Di sini dipilih versi dari cabang `fitur-judul`.

In [ ]:
tulis("judul.txt", "Judul: Catatan Belajar Git dan GitHub\n")
sh("cat judul.txt")
sh("git add judul.txt")
sh('git commit -m "Menyelesaikan konflik judul"')
sh("git log --oneline --graph --all")

## 7. Keamanan dan pengelolaan repositori

### a. Keamanan akun dan data
- **Autentikasi dua faktor (2FA)** menambah lapisan keamanan karena selain kata sandi diperlukan kode tambahan.
- **Kata sandi kuat dan unik**, jangan sama dengan layanan lain.
- **Menjaga token dan kunci SSH**: perlakukan seperti kata sandi, tidak dibagikan, tidak ditulis di dalam kode, dan segera dicabut (*revoke*) apabila bocor.
- **Tidak mengunggah data rahasia**: berkas seperti `.env`, kunci API, dan kata sandi basis data jangan di-commit. Daftarkan pada `.gitignore`.
- **Menangani kebocoran**: jika rahasia terlanjur terunggah, ganti atau cabut rahasia itu terlebih dahulu. Menghapus berkas pada commit berikutnya tidak cukup karena riwayat lama masih menyimpannya.

Berkas `.gitignore` berisi daftar berkas atau folder yang tidak perlu dilacak Git.

In [12]:
tulis(".gitignore", """# berkas rahasia
.env
# berkas sementara
*.log
__pycache__/
# folder hasil pengolahan
output/
""")

sh("cat .gitignore")
sh("git add .gitignore")
sh('git commit -m "Menambahkan .gitignore"')

$ cat .gitignore
# berkas rahasia
.env
# berkas sementara
*.log
__pycache__/
# folder hasil pengolahan
output/

$ git add .gitignore
fatal: not a git repository (or any of the parent directories): .git
[perintah berhenti dengan kode 128]

$ git commit -m "Menambahkan .gitignore"
fatal: not a git repository (or any of the parent directories): .git
[perintah berhenti dengan kode 128]



CompletedProcess(args='git commit -m "Menambahkan .gitignore"', returncode=128, stdout='', stderr='fatal: not a git repository (or any of the parent directories): .git\n')

### c. Membatalkan kesalahan

- Pada berkas yang belum di-commit, `git restore namaberkas` mengembalikan berkas ke isi commit terakhir.
- `git restore --staged namaberkas` mengeluarkan berkas dari staging area.
- Pada commit yang sudah dikirim ke GitHub, cara yang lebih aman adalah `git revert`, karena membuat commit baru yang membatalkan perubahan tanpa menghapus riwayat.
- `git reset --hard` **berbahaya** karena dapat menghapus perubahan secara permanen, sehingga harus dipakai dengan hati-hati. Perintah ini tidak dijalankan pada notebook ini.

Berikut `git restore` dan `git restore --staged` pada `catatan.txt`.

In [13]:
sh('echo "perubahan yang akan dibatalkan" >> catatan.txt')
sh("git status")
sh("git restore catatan.txt")
sh("git status")

sh('echo "perubahan yang akan dikeluarkan dari staging" >> catatan.txt')
sh("git add catatan.txt")
sh("git status")
sh("git restore --staged catatan.txt")
sh("git restore catatan.txt")
sh("git status")

$ echo "perubahan yang akan dibatalkan" >> catatan.txt

$ git status
fatal: not a git repository (or any of the parent directories): .git
[perintah berhenti dengan kode 128]

$ git restore catatan.txt
fatal: not a git repository (or any of the parent directories): .git
[perintah berhenti dengan kode 128]

$ git status
fatal: not a git repository (or any of the parent directories): .git
[perintah berhenti dengan kode 128]

$ echo "perubahan yang akan dikeluarkan dari staging" >> catatan.txt

$ git add catatan.txt
fatal: not a git repository (or any of the parent directories): .git
[perintah berhenti dengan kode 128]

$ git status
fatal: not a git repository (or any of the parent directories): .git
[perintah berhenti dengan kode 128]

$ git restore --staged catatan.txt
fatal: not a git repository (or any of the parent directories): .git
[perintah berhenti dengan kode 128]

$ git restore catatan.txt
fatal: not a git repository (or any of the parent directories): .git
[perintah berhenti de

CompletedProcess(args='git status', returncode=128, stdout='', stderr='fatal: not a git repository (or any of the parent directories): .git\n')

## 8. Menghubungkan Git dengan GitHub

Repositori lokal dan repositori di GitHub adalah dua tempat yang terpisah. Agar terhubung, repositori lokal perlu diberi tahu alamat repositori di GitHub yang disebut *remote*. Nama bawaan untuk remote utama adalah `origin`.

### Persiapan di GitHub, dilakukan sekali lewat peramban
1. Buat akun di github.com (**Sign up**) dan aktifkan autentikasi dua faktor.
2. Buat repositori baru: klik **+ → New repository**, isi **Repository name** `catatan-git-github`, pilih **Public** atau **Private**, jangan centang *Add a README file* (README.md sudah dibuat di sini), lalu **Create repository**.
3. Buat *personal access token*: **Settings → Developer settings → Personal access tokens**. Sesuai laporan, untuk operasi Git melalui HTTPS kata sandi akun tidak lagi dapat dipakai dan digantikan oleh token.

### Menyimpan token dengan aman
Disarankan menyimpan token pada **Secrets** Colab (ikon kunci 🔑 di panel kiri) dengan nama `GITHUB_TOKEN`, lalu aktifkan akses notebook. Jika tidak ada, token akan diminta lewat kolom tersembunyi. Token tidak ditulis di dalam kode maupun disimpan di konfigurasi Git. Cabut token di GitHub setelah selesai praktik.

Sel berikut meminta username, nama repositori (tekan Enter untuk memakai `catatan-git-github`), dan token. **Kosongkan username (Enter) jika belum ingin mengunggah**, maka bagian 8 dan 9 dilewati.

In [ ]:
from getpass import getpass


def tanya(pesan, rahasia=False):
    try:
        return (getpass(pesan) if rahasia else input(pesan)).strip()
    except EOFError:
        return ""


GH = {"aktif": False}

username = tanya("Username GitHub (kosongkan untuk melewati): ")
if not username:
    print("Dilewati. Bagian 8 dan 9 tidak dijalankan.")
else:
    nama_repo = tanya("Nama repositori [catatan-git-github]: ") or "catatan-git-github"

    token = None
    try:
        from google.colab import userdata
        token = userdata.get("GITHUB_TOKEN")
    except Exception:
        token = None
    if not token:
        token = tanya("Personal access token: ", rahasia=True)

    if not token:
        print("Token kosong. Bagian 8 dan 9 dilewati.")
    else:
        # Skrip kecil yang hanya mengembalikan token ketika Git meminta kata sandi
        askpass = Path(tempfile.mkdtemp()) / "askpass.sh"
        askpass.write_text('#!/bin/bash\necho "$GH_TOKEN"\n')
        askpass.chmod(askpass.stat().st_mode | stat.S_IEXEC)

        GH.update(
            aktif=True,
            username=username,
            repo=nama_repo,
            url=f"https://{username}@github.com/{username}/{nama_repo}.git",
            env={"GIT_ASKPASS": str(askpass), "GH_TOKEN": token},
            askpass=askpass,
        )
        token = None
        print(f"Akan memakai https://github.com/{username}/{nama_repo}")

### a. Menambahkan remote
- `git remote add origin ...` mendaftarkan alamat GitHub dengan nama `origin`.
- `git remote -v` menampilkan daftar remote untuk memastikan alamat sudah benar.
- `git branch -M main` mengganti nama cabang aktif menjadi `main` agar sama dengan GitHub.

### Mengirim ke GitHub
`git push -u origin main` mengirim commit ke cabang `main` pada remote `origin`. Opsi `-u` membuat Git mengingat tujuan ini sehingga selanjutnya cukup menulis `git push`. Setelah berhasil, berkas dapat dilihat pada halaman repositori di GitHub setelah halaman dimuat ulang.

In [ ]:
if not GH["aktif"]:
    print("Dilewati (tidak ada username/token).")
else:
    sh("git remote remove origin 2>/dev/null || true")
    sh(f"git remote add origin {shlex.quote(GH['url'])}")
    sh("git remote -v")
    sh("git branch -M main")
    sh("git push -u origin main", env=GH["env"])
    print(f"Lihat hasilnya di https://github.com/{GH['username']}/{GH['repo']}")

## 9. `git clone` dan `git pull`

**Git clone** dipakai untuk menyalin repositori yang sudah ada di GitHub ke komputer, lengkap dengan seluruh riwayatnya. Perintah ini membuat folder baru bernama sama dengan repositori, sudah berisi berkas dan riwayat, serta langsung mengatur `origin` sebagai remote, sehingga tidak memerlukan `git init` dan `git remote add`.

**Git pull** dipakai ketika repositori yang sama dikerjakan di dua tempat (misalnya komputer rumah dan komputer kampus, atau dua anggota kelompok). Apabila ada perubahan baru di GitHub, repositori lokal diperbarui dengan `git pull`. Pada dasarnya `git pull` adalah gabungan `git fetch` (mengambil data terbaru) dan penggabungan (*merge*). Kebiasaan yang baik adalah menjalankan `git pull` sebelum mulai bekerja dan sebelum `git push`.

Repositori yang baru diunggah disalin ke folder baru `catatan-git-github`, lalu diperbarui dengan `git pull`.

In [ ]:
if not GH["aktif"]:
    print("Dilewati (tidak ada username/token).")
else:
    salinan = BASE / GH["repo"]
    if salinan.exists():
        shutil.rmtree(salinan)
    sh(f"git clone {shlex.quote(GH['url'])}", cwd=BASE, env=GH["env"])
    sh("ls", cwd=salinan)
    sh("git log --oneline", cwd=salinan)
    sh("git pull origin main", cwd=salinan, env=GH["env"])

    GH["askpass"].unlink()     # hapus skrip token sementara
    GH["env"] = None

\## 10. Fork dan pull request

Pada repositori milik orang lain, pengguna biasanya tidak memiliki izin untuk mengubah langsung. Untuk berkontribusi dipakai alur *fork* dan *pull request*. **Fork** adalah salinan repositori milik orang lain yang disimpan di akun sendiri, sehingga bebas diubah tanpa memengaruhi repositori asli. **Pull request** adalah permintaan kepada pemilik repositori asli agar perubahan yang telah dibuat ditinjau dan digabungkan.

Langkah ini dilakukan lewat situs GitHub dan tidak dapat dijalankan dari notebook, sehingga hanya dicatat alurnya:

1. Klik tombol **Fork** pada repositori yang ingin dikontribusi.
2. Clone hasil fork ke komputer dengan `git clone`.
3. Buat cabang baru, misalnya `git switch -c perbaiki-typo`.
4. Lakukan perubahan, lalu jalankan `git add` dan `git commit`.
5. Kirim cabang ke GitHub dengan `git push origin perbaiki-typo`.
6. Pada GitHub, klik **Compare & pull request**, isi judul dan penjelasan perubahan, lalu klik **Create pull request**.
7. Pemilik repositori meninjau perubahan, dapat memberi komentar atau meminta perbaikan, kemudian menekan **Merge pull request** bila setuju.

Agar fork tetap sama dengan repositori asli, repositori asli ditambahkan sebagai remote tambahan bernama `upstream`:

```
git remote add upstream https://github.com/pemilik/proyek.git
git fetch upstream
git merge upstream/main
```

## 11. Ringkasan hasil pembelajaran

**Pengelompokan perintah Git menurut tahapan kerja

| Tahap | Perintah | Tujuan |
|---|---|---|
| Persiapan | `git config`, `git init`, `git clone` | Mengatur identitas dan memperoleh repositori |
| Pencatatan | `git status`, `git add`, `git commit` | Memeriksa, memilih, dan menyimpan perubahan |
| Pemeriksaan | `git log`, `git diff` | Melihat riwayat dan perbedaan |
| Percabangan | `git branch`, `git switch`, `git merge` | Bekerja di jalur terpisah lalu menggabungkan |
| Sinkronisasi | `git remote`, `git push`, `git pull`, `git fetch` | Menghubungkan dan menyamakan dengan GitHub |
| Perbaikan | `git restore`, `git revert` | Membatalkan kesalahan dengan aman |

**Dua cara mengunggah berkas :** melalui peramban (**Add file → Upload files**, cocok untuk berkas sedikit) dan melalui Git di komputer dengan urutan `git add`, `git commit`, `git push` (lebih sesuai untuk pekerjaan berkelanjutan karena riwayat tercatat rapi). Notebook ini memakai cara kedua.

**Hal yang diperhatikan sebelum mengunggah:** memastikan alamat remote benar, melakukan autentikasi dengan token atau SSH, memastikan tidak ada berkas rahasia yang ikut terunggah, serta menulis pesan commit yang bermakna.

Riwayat akhir repositori hasil praktik:

In [ ]:
sh("git log --oneline --graph --all")
sh("git status")
sh("ls -a")